# Discovering disruptions of non-linear correlations

Shows how non-linear correlation disruption can be identified.

**IMPORTANT: Run this notebook LAST!**

This notebook requires the following extra packages: `dcor==0.6`, `hyppo==0.5.2`, `maxcorr==0.1.2`, `minepy==1.2.6`.

Note that this will result in other packages to change version number, potentially implicating some of the reproduction results in the other notebooks. So we would recommend **running this notebook last.**

In [ ]:
%%bash
# install packages mentioned above
micromamba install -n nedis -y "minepy==1.2.6" "dcor==0.6" "numba==0.60.0" "numpy==1.26.4" "scipy==1.13.1" "autograd==1.8.0" "future==1.0.0"
pip install --no-deps "hyppo==0.5.2" "maxcorr==0.1.2"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(16)

In [ ]:
from pathlib import Path
import numpy as np
import scipy
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from nedis.visualization import visualize_data

In [ ]:
random_state = 43

out_path = Path("../_out/synthetic")
out_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def corr_mat(X, Y=None, kind="pearson", **params):
    """
    Calculates a correlation matrix with different measures inclusing non-linear ones.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        The input samples.
    Y : array-like, shape (n_samples, n_features), optional
        The target samples. If None, the correlation matrix is calculated for X.
    kind : str, optional
        The kind of correlation to calculate. Supported values are:
        - 'pearson': Pearson correlation coefficient
        - 'spearman': Spearman rank correlation coefficient
        - 'mic': Maximal Information Coefficient (MIC)
        - 'miclog': -log(MIC)
        - 'mi': Mutual Information (MI)
        - 'dcor': Distance Correlation
        - 'hsic': Hilbert-Schmidt Independence Criterion (HSIC)
        - 'crosscorr': Cross-correlation
        - 'maxcorr': Maximal Correlation (HGR)
        Default is 'pearson'.
    **params : dict, optional
        Additional parameters to pass to the correlation function. For example, for 'mic', you can pass {'alpha': 0.6} to set the alpha parameter of the MINE estimator.

    Returns
    -------
    C : array, shape (n_features, n_features)
        The correlation matrix.
    """

    if Y is not None:
        raise NotImplementedError("Y must be None at the moment")
    
    if kind == 'mic' or kind == 'miclog':
        from minepy import MINE
    elif kind == 'mi':
        from sklearn.feature_selection import mutual_info_regression
    elif kind == 'pearson':
        from scipy.stats import pearsonr
    elif kind == 'spearman':
        from scipy.stats import spearmanr
    elif kind == 'dcor':
        import dcor
    elif kind == "hsic":
        from hyppo.independence import Hsic
    elif kind == "crosscorr":
        from scipy import signal
    elif kind == "maxcorr":
        from maxcorr import indicator
    else:
        raise ValueError(f"Unknown correlation kind: {kind}")
            
    
    if params is None:
        params = {}
    p = X.shape[1]
    C = np.eye(p)
    for i in range(p):
        for j in range(i+1, p):
            if kind == "pearson":
                c = pearsonr(X[:, i], X[:, j], **params)[0]
            elif kind == "spearman":
                c = spearmanr(X[:, i], X[:, j], **params).correlation
            elif kind == "mic":
                m = MINE(**params)
                m.compute_score(X[:, i], X[:, j])
                c = m.mic()
                # c -= 0.1
                # c *= 1 / 0.6
                # c = max(0., c)
                # c = min(1., c)
            elif kind == "miclog":
                m = MINE(**params)
                m.compute_score(X[:, i], X[:, j])
                c = -np.log(m.mic())
            elif kind == "mi":
                c = mutual_info_regression(
                    X[:, i].reshape(-1, 1), X[:, j], **params
                )[0]
                # normalize to [0,1]
                c /= np.log(X.shape[0])
                # c = max(0., c)
                # c = min(1., c)
            elif kind == "dcor":
                c = dcor.distance_correlation(X[:, i], X[:, j])
            elif kind == "hsic":
                c = 1 - Hsic().test(X[:, i].reshape(-1,1), X[:, j].reshape(-1,1), **params)[1]
            elif kind == "crosscorr":
                x0 = (X[:, i] - X[:, i].mean())/X[:, i].std()
                y0 = (X[:, j] - X[:, j].mean())/X[:, j].std()
                corr = signal.correlate(x0, y0, mode='full', method='fft')
                corr_norm = corr / len(x0)
                c = np.max(corr_norm)
            elif kind == "maxcorr":
                est = indicator(semantics='hgr', algorithm='dk', backend='numpy')
                c = est.compute(X[:, i], X[:, j])
            else:
                raise ValueError(f"Unknown correlation kind: {kind}")
            C[i, j] = C[j, i] = c
    return C

In [ ]:
def analyse_variable_relationships(X, correlations=["spearman", "mic", "mi", "dcor", "hsic", "crosscorr"], ylogscale=False):
    """
    Analyzes the relationships between variables in X using different correlation measures.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        The input samples.
    correlations : list of str, optional
        The list of correlation measures to analyze. 
        See the `corr_mat` function for supported measures. Default is ["spearman", "mic", "mi", "dcor", "hsic", "crosscorr"].
    ylogscale : bool, optional
        Whether to use a logarithmic scale for the y-axis in the histogram. Default is False

    Returns
    -------
    matrices : dict
        A dictionary where the keys are the correlation measures and the values are the corresponding correlation matrices.
    """

    fig, ax = plt.subplots(2, len(correlations), figsize=(5 * len(correlations), 8))
    matrices = {}
    for i, corr_kind in enumerate(correlations):
        C = corr_mat(X, kind=corr_kind)
        sns.heatmap(
            pd.DataFrame(np.abs(C)), 
            center=0, 
            vmin=-1, 
            vmax=1, 
            ax=ax[0, i])
        sns.histplot(C.flatten(), bins=np.linspace(-1, 1, 50), ax=ax[1, i])
        ax[0, i].set_title(corr_kind)
        if ylogscale:
            ax[1, i].set_yscale("log")
        matrices[corr_kind] = C

    return matrices

In [ ]:
# Functions to perturb the data by swapping values within a certain distance, 
# which can be used to test the sensitivity of the correlation measures to non-linear relationships.

def swap(arr, arr2=None, distance=0.1):
    if arr2 is None:
        arr2 = arr
    i = np.random.randint(0, len(arr))
    js = [j for j in range(len(arr)) if np.abs(arr[i] - arr2[j]) <= distance]
    j = np.random.choice(js)
    arr[i], arr[j] = arr[j], arr[i]

def perturb_array(arr, arr2=None, n_swaps=10, distance=0.1):
    if arr2 is None:
        arr2 = arr
    arr_perturbed = arr.copy()
    for _ in range(n_swaps):
        swap(arr_perturbed, distance=distance)
    return arr_perturbed

def perturb_matrix(X, n_swaps=None, distance=0.1):
    if n_swaps is None:
        n_swaps = X.shape[0] * 2
    X_perturbed = X.copy()
    for i in range(X.shape[1]):
        X_perturbed[:, i] = perturb_array(
            X[:, i], n_swaps=n_swaps, distance=distance
        )
    return X_perturbed

In [ ]:
# Example usage of the perturbation functions to test the sensitivity of correlation measures.

x0 = np.random.uniform(-1, 1, size=100)
y0 = x0

y_perturbed1 = perturb_array(y0, x0, n_swaps=200, distance=0.2)
y_perturbed2 = perturb_array(y0, x0, n_swaps=200, distance=2)
plt.scatter(x0, y0)
plt.scatter(x0, y_perturbed1)
plt.scatter(x0, y_perturbed2)

r, p = scipy.stats.spearmanr(x0, y0)
r1, p1 = scipy.stats.spearmanr(x0, y_perturbed1)
r2, p2 = scipy.stats.spearmanr(x0, y_perturbed2)
r, r1, r2

In [ ]:
# To get a more robust estimate of the correlation values, 
# we can repeat the perturbation and correlation calculation multiple times 
# and analyze the distribution of the correlation coefficients.

x0 = np.random.uniform(-1, 1, size=100)
y0 = x0

rs = []
for i in tqdm(range(100)):
    y_perturbed1 = perturb_array(y0, x0, n_swaps=200, distance=0.7)
    y_perturbed2 = perturb_array(y0, x0, n_swaps=200, distance=2)

    r, p = scipy.stats.spearmanr(x0, y0)
    r1, p1 = scipy.stats.spearmanr(x0, y_perturbed1)
    r2, p2 = scipy.stats.spearmanr(x0, y_perturbed2)
    rs.append((r, r1, r2))

rs = np.array(rs)
plt.boxplot(rs, positions=[0, 1, 2])
plt.xticks([0, 1, 2], ["original", "perturbed close", "perturbed far"])

print("r=", np.mean(rs[:,0], axis=0))
print("r1=", np.mean(rs[:,1], axis=0))
print("r2=", np.mean(rs[:,2], axis=0))



In [ ]:
# Function for generating a block of variables with a non-linear relationship (sinusoidal) 
# that creates clusters in the scatter plot (non-linear correlation) but has low Pearson/Spearman correlation.

def generate_block_sinusoid_sincos(
        n=100, m=5, random_state=None):
    rng = np.random.default_rng(random_state)
    theta = rng.uniform(0, 4*np.pi, size=n)

    Xblock = np.column_stack(
        [np.sin(theta) for i in range(int(np.ceil(m / 2)))] +
        [np.cos(theta) for i in range(m // 2)]
    )

    return Xblock

X = generate_block_sinusoid_sincos(
    n=100, m=6, random_state=42)
X = np.concatenate([X, np.random.normal(size=(100, 4))], axis=1)
analyse_variable_relationships(X);

sns.pairplot(pd.DataFrame(X))

# Other ways to generate non-linear correlation clusters that change over time

# Variant 1: Sinusoidal block with different frequencies (wavelengths)

# def generate_block_sinusoid_sincos_wavelength(
#         n=2000, m=5, wavelength_step=1, random_state=None):
#     rng = np.random.default_rng(random_state)
#     theta = rng.uniform(0, 2*np.pi, size=n)

#     # sinusoidal block of size m
#     Xblock = np.column_stack(
#         [np.sin((i*wavelength_step+1)*theta) for i in range(int(np.ceil(m / 2)))] +
#         [np.cos((i*wavelength_step+1)*theta) for i in range(m // 2)]
#     )

#     return Xblock

# X = generate_block_sinusoid_sincos_wavelength(
#     n=100, m=6, wavelength_step=2, random_state=33)
# X = np.concatenate([X, np.random.normal(size=(100, 4))], axis=1)
# analyse_variable_relationships(X);

# sns.pairplot(pd.DataFrame(X))

# Variant 2: Non-linear block with independent sign flips to create zero Pearson correlation but high non-linear correlation

# def generate_block_nonlinear_zero_corr(
#         n=2000, m=5, base_fn=None, random_state=None):
#     """
#     Generate m variables with high non-linear correlation (via shared base_fn)
#     but near-zero Pearson correlation via independent sign flips.
#     """
#     rng = np.random.default_rng(random_state)
#     # latent angle uniformly distributed; you could use other symmetric distributions

#     if base_fn is None:
#         base_fn = lambda x: x
#     theta = rng.uniform(0, 2*np.pi, size=n)
#     base = base_fn(theta)         # common nonlinear signal
    
#     Xblock = np.zeros((n, m))
#     for j in range(m):
#         # independent ±1 sign flips for each variable and each sample
#         signs = rng.choice([-1, 1], size=n)
#         Xblock[:, j] = base * signs
    
#     return Xblock

# X = generate_block_nonlinear_zero_corr(
#     n=100, m=6, base_fn=lambda x:x, random_state=42)
# X = np.concatenate([X, np.random.normal(size=(100, 4))], axis=1)
# analyse_variable_relationships(X);

# sns.pairplot(pd.DataFrame(X))

In [ ]:
n = 100
n_steps = 5
noise_levels = np.array([0.01, 0.05, 0.10, 0.50, 1.00]) * 2
# noise_levels = np.array([0.01, 0.025, 0.5, 0.75, 1.0]) * 5
# noise_levels = np.array([0.001, 0.05, 0.1, 0.2, 0.3]) * 2
noise_levels = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
Xs = []
correlation_matrices = []
for i in range(n_steps):

    X1 = generate_block_sinusoid_sincos(n=n, m=20, random_state=43)
    X2 = generate_block_sinusoid_sincos(n=n, m=20, random_state=66)

    # noise
    X1 = perturb_matrix(X1, distance=noise_levels[i], n_swaps=3* n)
    X2 = perturb_matrix(X2, distance=noise_levels[i], n_swaps=3* n)

    # X1 += np.random.default_rng(42).normal(size=X1.shape) * noise_levels[i]
    # X2 += np.random.default_rng(99).normal(size=X2.shape) * noise_levels[i]

    X3 = np.random.default_rng(99).normal(size=(n, 20))  # independent noise variables
    X = np.column_stack([X1, X3, X2])
    Xs.append(X)
    cms = analyse_variable_relationships(X, ylogscale=True)
    correlation_matrices.append(cms)

    plt.show()

X = np.concatenate(Xs, axis=0)
y0 = np.repeat(range(n_steps), repeats=n)
entities = np.tile(range(n), n_steps)

In [ ]:
# visualize
for corr in ["spearman", "mic"]:
    fig, axes, _, coordinates = visualize_data(
        X, y0, entities, 
        # coordinates=np.array([
        #     (0,0),(0,1),(1,0),(1,1), 
        #     (2,2),(2,3),(3,2),(3,3), 
        #     (4,4),(4,5),(5,4),(5,5)]),
        tsne_perplexity=30,
        mode="network", 
        correlation_threshold=0.4,
        correlation_matrices=[cm[corr] for cm in correlation_matrices],
        random_state=random_state);
    fig.suptitle(f"Correlation: {corr}")
    fig.savefig(out_path / f"non-linear_correlation_{corr}.png", bbox_inches='tight')
    fig.savefig(out_path / f"non-linear_correlation_{corr}.pdf", bbox_inches='tight')

In [ ]:
from nedis.cluster.leidenalg import WeightedLeidenClustering
clustering = WeightedLeidenClustering(resolution_parameter=1.1)
clustering.fit(correlation_matrices[0]["mic"])
clustering.labels_

In [ ]:
from nedis.cordis.default \
    import DefaultCorrelationDisruptionFeatureTransformer \
    as DefaultCorrelationDisruptionTransformer

In [ ]:
# configure and fir correlation disruption transformer
disruption_transformer = DefaultCorrelationDisruptionTransformer(
    default_clustering_correlation_function=lambda X: corr_mat(X, kind="mic"),
    default_clustering_resolution_parameter=1.1,
    default_optimization_separation_score="spearman",
    default_optimization=False,
    default_derive_features_aggregation="mean")
disruption_transformer.fit(X, y0, groups=entities, subset_masks="y", 
    reference_labels=np.array([0]));

In [ ]:
# transform samples
disruption_values = disruption_transformer.transform(X)

In [ ]:
# we get one aggregated disruption value per cluster for each sample
disruption_values.shape

In [ ]:
import numpy as np
import scipy.stats

import matplotlib.pyplot as plt
import seaborn as sns

from nedis.visualization import plot_cordis_cluster as plot_cluster

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
theta = np.random.uniform(0, 4*np.pi, size=n)
ax = axes[0]
ax.scatter(theta, np.sin(theta))
ax.scatter(theta, np.cos(theta))
ax.set_xlabel("theta")
# ax.set_ylabel("Variable 5")
# ax.set_title("Linear relationship")
ax.legend(["sin(theta)", "cos(theta)"])
ax = axes[1]
ax.scatter(X[y0 == 0, 0], X[y0 == 0, 4])
ax.set_xlabel("Variable 1 (sin)")
ax.set_ylabel("Variable 5 (sin)")
ax.set_title("Linear relationship")
ax = axes[2]
ax.scatter(X[y0 == 1, 0], X[y0 == 1, 14], color='C1')
ax.set_xlabel("Variable 1 (sin)")
ax.set_ylabel("Variable 15 (cos)")
ax.set_title("Non-linear relationship")

fig.savefig(out_path / "non-linear_scatterplots.png", bbox_inches='tight')
fig.savefig(out_path / "non-linear_scatterplots.pdf", bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
y_unique = np.unique(y0)

for i_cluster, cluster in enumerate(disruption_transformer.selected_clusters_):
    
    values = disruption_values[:, i_cluster]
    r, p = scipy.stats.spearmanr(values, y0)
    
    fig, axes = plt.subplots(
        1, len(y_unique) + 1, 
        figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), 
        dpi=300)
 
    # correlation disruption plot
    ax = axes[0]
    x_rank = scipy.stats.rankdata(y0, method="dense")
    sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=values, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption",
        title=f"r={r:.02f}"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    # cluster visualization
    for i, yy in enumerate(y_unique):
        ax = axes[i + 1]
        ax.axis("off")
        plot_cluster(
            cluster, 
            coordinates, 
            correlation_matrices[yy]["mi"], 
            correlation_threshold=0, 
            verbose=0,
            ax=ax)
    fig.suptitle(
        f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    fig.savefig(out_path / f"non-linear_cluster_{i_cluster}_ref{cluster['reference_label']}_id_{'-'.join(str(c) for c in cluster['id'])}.png", bbox_inches='tight')
    fig.savefig(out_path / f"non-linear_cluster_{i_cluster}_ref{cluster['reference_label']}_id_{'-'.join(str(c) for c in cluster['id'])}.pdf", bbox_inches='tight')
    plt.show()

    if i_cluster >= 10:
        print(f"Breaking for demo purposes (remaining clusters: {len(disruption_transformer.selected_clusters_) - i_cluster - 1}) / {len(disruption_transformer.selected_clusters_)}")
        break
    

## Examining invidual disruption values under non-linear relationships

**Conclusion:** While non-linear relationships can be picked up by NeDis they are brittle and disruptions of such non-linear relationships are harder to detect than disruptions of linear relationships. 

In [ ]:
import numpy as np
from maxcorr import indicator

n = 5000
theta = np.random.rand(n) * 2 * np.pi
theta2 = np.random.rand(n) * 2 * np.pi
x = np.sin(theta)
y = x #np.cos(theta)
y = np.random.uniform(size=n)


est = indicator(semantics='hgr', algorithm='dk', backend='numpy')
score = est.compute(x, y)

print("HGR maximal correlation (sin vs cos):", score) 


In [ ]:
from random import sample
from tqdm import tqdm
from sklearn.feature_selection import mutual_info_regression
from minepy import MINE

# suppress deprecation warnings from minepy
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

n_runs = 10

noise_levels2 = np.array([0.01, 0.025, 0.5, 0.75, 1.0, 1.25, 1.5]) * 5

for i_noise, noise_level in enumerate(noise_levels2):

    disruption_values1_spearman_means = np.zeros(n_runs)
    disruption_values2_spearman_means = np.zeros(n_runs)
    disruption_values3_spearman_means = np.zeros(n_runs)
    disruption_values1_mic_means = np.zeros(n_runs)
    disruption_values2_mic_means = np.zeros(n_runs)
    disruption_values3_mic_means = np.zeros(n_runs)
    disruption_values1_mi_means = np.zeros(n_runs)
    disruption_values2_mi_means = np.zeros(n_runs)
    disruption_values3_mi_means = np.zeros(n_runs)

    for j in tqdm(range(n_runs)):

        n = 100
        theta = np.linspace(0, 2*np.pi, n)
        x0 = np.sin(theta)# + np.random.normal(scale=0.1, size=theta.shape)
        x1 = np.sin(theta)# + np.random.normal(scale=0.1, size=theta.shape)
        x2 = np.cos(theta)# + np.random.normal(scale=0.1, size=theta.shape)
        x3 = x0 * np.random.choice([-1, 1], size=n)  # sign-flipped version of x1

        # noise = 0.5
        # n_r = 100
        # x1r = np.random.uniform(-1, 1, size=n_r) + np.random.normal(scale=noise, size=n_r)
        # x2r = np.random.uniform(-1, 1, size=n_r) + np.random.normal(scale=noise, size=n_r)
        # x3r = np.random.uniform(-1, 1, size=n_r) + np.random.normal(scale=noise, size=n_r)
        # x0r = x0 + noise_level * np.random.normal(size=n)
        # x1r = x1 + noise_level * np.random.normal(size=n)
        # x2r = x2 + noise_level * np.random.normal(size=n)
        # x3r = x3 + noise_level * np.random.normal(size=n)

        x0r = perturb_array(x0, n_swaps=300, distance=noise_level)
        x1r = perturb_array(x1, n_swaps=300, distance=noise_level)
        x2r = perturb_array(x2, n_swaps=300, distance=noise_level)
        x3r = perturb_array(x3, n_swaps=300, distance=noise_level)
        n_r = x1r.shape[0]

        mine = MINE()
        mine.compute_score(x0, x1)
        c1_mic = mine.mic()
        mine.compute_score(x0, x2)
        c2_mic = mine.mic()
        mine.compute_score(x0, x3)
        c3_mic = mine.mic()

        c1_spearman, _ = scipy.stats.spearmanr(x0, x1)
        c2_spearman, _ = scipy.stats.spearmanr(x0, x2)
        c3_spearman, _ = scipy.stats.spearmanr(x0, x3)

        c1_mi = mutual_info_regression(x0.reshape(-1, 1), x1)
        c2_mi = mutual_info_regression(x0.reshape(-1, 1), x2)
        c3_mi = mutual_info_regression(x0.reshape(-1, 1), x3)

        disruption_values1_spearman = np.zeros(n_r)
        disruption_values2_spearman = np.zeros(n_r)
        disruption_values3_spearman = np.zeros(n_r)
        disruption_values1_mic = np.zeros(n_r)
        disruption_values2_mic = np.zeros(n_r)
        disruption_values3_mic = np.zeros(n_r)
        disruption_values1_mi = np.zeros(n_r)
        disruption_values2_mi = np.zeros(n_r)
        disruption_values3_mi = np.zeros(n_r)

        for i in range(n_r):

            # sample_idx = np.random.choice(n, size=5, replace=False)
            sample_idx = [i]

            r, p = scipy.stats.spearmanr(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x1, x1r[sample_idx]]))
            disruption_values1_spearman[i] = r - c1_spearman

            r, p = scipy.stats.spearmanr(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x2, x2r[sample_idx]]))
            disruption_values2_spearman[i] = r - c2_spearman

            r, p = scipy.stats.spearmanr(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x3, x3r[sample_idx]]))
            disruption_values3_spearman[i] = r - c3_spearman

            mine.compute_score(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x1, x1r[sample_idx]]))
            c_r = mine.mic()
            disruption_values1_mic[i] = c_r - c1_mic

            mine.compute_score(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x2, x2r[sample_idx]]))
            c_r = mine.mic()
            disruption_values2_mic[i] = c_r - c2_mic

            mine.compute_score(
                np.concatenate([x0, x0r[sample_idx]]), 
                np.concatenate([x3, x3r[sample_idx]]))
            c_r = mine.mic()
            disruption_values3_mic[i] = c_r - c3_mic

            mi1 = mutual_info_regression(
                np.concatenate([x0, x0[sample_idx]]).reshape(-1, 1), 
                np.concatenate([x1, x1r[sample_idx]]))
            disruption_values1_mi[i] = mi1 - c1_mi 

            mi2 = mutual_info_regression(
                np.concatenate([x0, x0[sample_idx]]).reshape(-1, 1), 
                np.concatenate([x2, x2r[sample_idx]]))
            disruption_values2_mi[i] = mi2 - c2_mi

            mi3 = mutual_info_regression(
                np.concatenate([x0, x0[sample_idx]]).reshape(-1, 1), 
                np.concatenate([x3, x3r[sample_idx]]))
            disruption_values3_mi[i] = mi3 - c3_mi  

        if (i_noise == 0 and j == 0) or (i_noise == len(noise_levels2) - 1 and j == n_runs - 1):
            fig, ax = plt.subplots(1, 3, figsize=(14, 4))
            sns.scatterplot(x=x0, y=x1, ax=ax[0])
            ax[0].set_title("x1 vs x2 (sin-sin)")
            sns.scatterplot(x=x0, y=x2, ax=ax[1])
            ax[1].set_title("x1 vs x3 (sin-cos)")
            sns.scatterplot(x=x0, y=x3, ax=ax[2])
            ax[2].set_title("x1 vs x4 (sin-sin sign-flipped)")

            ax[0].scatter(x0r, x1r, color="red", marker="x")
            ax[1].scatter(x0r, x2r, color="red", marker="x")
            ax[2].scatter(x0r, x3r, color="red", marker="x")

            fig, ax = plt.subplots(1, 3, figsize=(10, 4), sharex=False, sharey=False)
            sns.histplot(disruption_values1_spearman, bins=20, ax=ax[0])
            sns.histplot(disruption_values2_spearman, bins=20, ax=ax[1])
            sns.histplot(disruption_values3_spearman, bins=20, ax=ax[2])
            ax[0].axvline(np.mean(disruption_values1_spearman), color="red", linestyle="--")
            ax[1].axvline(np.mean(disruption_values2_spearman), color="red", linestyle="--")
            ax[2].axvline(np.mean(disruption_values3_spearman), color="red", linestyle="--")
            ax[0].axvline(0, color="black", linestyle=":")
            ax[1].axvline(0, color="black", linestyle=":")
            ax[2].axvline(0, color="black", linestyle=":")
            fig.suptitle("Spearman correlation disruption")
            plt.show()

            fig, ax = plt.subplots(1, 3, figsize=(10, 4), sharex=False, sharey=False)
            sns.histplot(disruption_values1_mic, bins=20, ax=ax[0])
            sns.histplot(disruption_values2_mic, bins=20, ax=ax[1])
            sns.histplot(disruption_values3_mic, bins=20, ax=ax[2])
            ax[0].axvline(np.mean(disruption_values1_mic), color="red", linestyle="--")
            ax[1].axvline(np.mean(disruption_values2_mic), color="red", linestyle="--")
            ax[2].axvline(np.mean(disruption_values3_mic), color="red", linestyle="--")
            ax[0].axvline(0, color="black", linestyle=":")
            ax[1].axvline(0, color="black", linestyle=":")
            ax[2].axvline(0, color="black", linestyle=":")
            fig.suptitle("MIC disruption")
            plt.show()

            fig, ax = plt.subplots(1, 3, figsize=(10, 4), sharex=False, sharey=False)
            sns.histplot(disruption_values1_mi, bins=20, ax=ax[0])
            sns.histplot(disruption_values2_mi, bins=20, ax=ax[1])
            sns.histplot(disruption_values3_mi, bins=20, ax=ax[2])
            ax[0].axvline(np.mean(disruption_values1_mi), color="red", linestyle="--")
            ax[1].axvline(np.mean(disruption_values2_mi), color="red", linestyle="--")
            ax[2].axvline(np.mean(disruption_values3_mi), color="red", linestyle="--")
            ax[0].axvline(0, color="black", linestyle=":")
            ax[1].axvline(0, color="black", linestyle=":")
            ax[2].axvline(0, color="black", linestyle=":")
            fig.suptitle("Mutual Information disruption")
            plt.show()

        disruption_values1_spearman_means[j] = np.mean(disruption_values1_spearman)
        disruption_values2_spearman_means[j] = np.mean(disruption_values2_spearman)
        disruption_values3_spearman_means[j] = np.mean(disruption_values3_spearman)
        disruption_values1_mic_means[j] = np.mean(disruption_values1_mic)
        disruption_values2_mic_means[j] = np.mean(disruption_values2_mic)
        disruption_values3_mic_means[j] = np.mean(disruption_values3_mic)
        disruption_values1_mi_means[j] = np.mean(disruption_values1_mi)
        disruption_values2_mi_means[j] = np.mean(disruption_values2_mi)
        disruption_values3_mi_means[j] = np.mean(disruption_values3_mi)


    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=False, sharey=False)

    ax = axes[0]
    sns.boxplot(data=[
        disruption_values1_spearman_means,
        disruption_values2_spearman_means,
        disruption_values3_spearman_means], ax=ax)
    ax.axhline(0, color="black", linestyle=":")
    ax.set_title("Disruption values (Spearman) means over runs")
    ax.set_ylabel("Disruption value mean")

    ax = axes[1]
    sns.boxplot(data=[
        disruption_values1_mic_means,
        disruption_values2_mic_means,
        disruption_values3_mic_means], ax=ax)
    ax.axhline(0, color="black", linestyle=":")
    ax.set_title("Disruption values (MIC) means over runs")
    ax.set_ylabel("Disruption value mean")

    ax = axes[2]
    sns.boxplot(data=[
        disruption_values1_mi_means,
        disruption_values2_mi_means,
        disruption_values3_mi_means], ax=ax)
    ax.axhline(0, color="black", linestyle=":")
    ax.set_title("Disruption values (Mutual Information) means over runs")
    ax.set_ylabel("Disruption value mean")

    plt.show()

    # fig, ax = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
    # sns.histplot(disruption_values1_spearman_means, bins=20, ax=ax[0])
    # sns.histplot(disruption_values2_spearman_means, bins=20, ax=ax[1])
    # sns.histplot(disruption_values3_spearman_means, bins=20, ax=ax[2])
    # for a in ax:
    #     a.axvline(0, color="black", linestyle=":")

    # fig, ax = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
    # sns.histplot(disruption_values1_mic_means, bins=20, ax=ax[0])
    # sns.histplot(disruption_values2_mic_means, bins=20, ax=ax[1])
    # sns.histplot(disruption_values3_mic_means, bins=20, ax=ax[2])
    # for a in ax:
    #     a.axvline(0, color="black", linestyle=":")  
    
    # fig, ax = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
    # sns.histplot(disruption_values1_mi_means, bins=20, ax=ax[0])
    # sns.histplot(disruption_values2_mi_means, bins=20, ax=ax[1])
    # sns.histplot(disruption_values3_mi_means, bins=20, ax=ax[2])
    # for a in ax:
    #     a.axvline(0, color="black", linestyle=":")
